Objective: Improve AdventureWorks’ customer retention, product performance, and inventory planning by analyzing multi-channel sales patterns using SQL set operators.

P1 <span style="color: var(--vscode-foreground);">Customers who made purchases online </span> _or_ <span style="color: var(--vscode-foreground);">in-store</span>

Specs: <span style="color: var(--vscode-foreground);">Retrieve all CustomerIDs from SalesOrderHeader&nbsp;where OnlineOrderFlag is 1 (online orders).&nbsp;</span>  <span style="color: var(--vscode-foreground);">Retrieve all CustomerID where OnlineOrderFlag&nbsp;is 0 (in-store orders).&nbsp;</span>  <span style="color: var(--vscode-foreground);">Using UNION to merge both sets and remove duplicates.</span>

Reason: Gives a full picture of all active customers across both sales channels.

In [ ]:
USE AdventureWorks2022;
GO
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 1
UNION
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 0;

P2 <span style="color: var(--vscode-foreground);">Customers who shop </span> _both_ <span style="color: var(--vscode-foreground);">online and in-store</span>

<span style="color: var(--vscode-foreground);">Spec:&nbsp;</span> <span style="color: var(--vscode-foreground);">Select customers who bought online and who bought in-store.&nbsp;</span> <span style="color: var(--vscode-foreground);">Using INTERSECT</span> <span style="color: var(--vscode-foreground);">&nbsp;to return only those present in both result sets.</span>

<span style="color: var(--vscode-foreground);">Reason: Identifies loyal, multi-channel shoppers for cross-promotion opportunities.</span>

In [ ]:
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 1
INTERSECT
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 0;


P3: <span style="color: var(--vscode-foreground);">Find customers who placed online orders but never bought in-store</span>

Specs: <span style="color: var(--vscode-foreground);">Select all customers who shopped online (</span><span style="color: var(--vscode-foreground);">OnlineOrderFlag=1)</span>  <span style="color: var(--vscode-foreground);">&nbsp;Subtract those who shopped in store (OnlineOrderFlag=0) using EXCEPT.&nbsp;</span>  <span style="color: var(--vscode-foreground);">The result is online-only customers.</span>

Reason: <span style="color: var(--vscode-foreground);">Helps target digital buyers with online-exclusive deals and marketing.</span>

In [ ]:
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 1
EXCEPT
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 0;


P4: <span style="color: var(--vscode-foreground);">The customers who shop in person but never online.</span>

Specs: <span style="color: var(--vscode-foreground);">Select all customers who shopped in-store (</span><span style="color: var(--vscode-foreground);">(OnlineOrderFlag=0)</span><span style="color: var(--vscode-foreground);">.&nbsp;</span>    <span style="color: var(--vscode-foreground);">Subtract those who shopped online (</span><span style="color: var(--vscode-foreground);">OnlineOrderFlag=1</span><span style="color: var(--vscode-foreground);">) using EXCEPT.&nbsp;</span>  <span style="color: var(--vscode-foreground);">The result is in-store-only customers.</span>

Reason: Shows who prefers in-person shopping to improve store experience or drive app sign-ups.

In [ ]:
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 0
EXCEPT
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 1;

P5: Products that sold in the latest two sales years

Specs: Identify the two most recent sales years from SalesOrderHeader.OrderDate. Pull product IDs from SalesOrderDetail for those years. Combine both sets with UNION ALL, then use DISTINCT to remove duplicates. 

Reason: Reveals consistently popular products that stay in demand year to year.

In [ ]:
SELECT DISTINCT ProductID
FROM Sales.SalesOrderDetail sod
JOIN Sales.SalesOrderHeader soh 
    ON soh.SalesOrderID = sod.SalesOrderID
WHERE YEAR(soh.OrderDate) = 2013
UNION
SELECT DISTINCT ProductID
FROM Sales.SalesOrderDetail sod
JOIN Sales.SalesOrderHeader soh 
    ON soh.SalesOrderID = sod.SalesOrderID
WHERE YEAR(soh.OrderDate) = 2014;


P6: Customers who bought both Bikes and Accessories
Specs: Uses INTERSECT to return customers who appear in both sets.

Reason: Highlights cross-selling success and customers likely to buy add-ons.

In [ ]:
SELECT DISTINCT soh.CustomerID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod 
    ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product p 
    ON sod.ProductID = p.ProductID
JOIN Production.ProductSubcategory s 
    ON p.ProductSubcategoryID = s.ProductSubcategoryID
JOIN Production.ProductCategory c 
    ON s.ProductCategoryID = c.ProductCategoryID
WHERE c.Name = 'Bikes'
INTERSECT
SELECT DISTINCT soh.CustomerID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod 
    ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product p 
    ON sod.ProductID = p.ProductID
JOIN Production.ProductSubcategory s 
    ON p.ProductSubcategoryID = s.ProductSubcategoryID
JOIN Production.ProductCategory c 
    ON s.ProductCategoryID = c.ProductCategoryID
WHERE c.Name = 'Accessories';

P7 Products that are sold online and also to reseller stores.

<span style="color: var(--vscode-foreground);">Specs:&nbsp;</span>  <span style="color: var(--vscode-foreground);">Use INTERSECT&nbsp;</span>  between online and in-store products

Reason: Finds top-performing SKUs that succeed across multiple channels.

In [ ]:
SELECT DISTINCT sod.ProductID
FROM Sales.SalesOrderDetail sod
JOIN Sales.SalesOrderHeader soh 
    ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 1
INTERSECT
SELECT DISTINCT sod.ProductID
FROM Sales.SalesOrderDetail sod
JOIN Sales.SalesOrderHeader soh 
    ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 0;

P8 Active catalog items that didn’t sell in 2014.

<span style="color: var(--vscode-foreground);">Spec:&nbsp;</span> <span style="color: var(--vscode-foreground);">EXCEPT compares</span> <span style="color: var(--vscode-foreground);">&nbsp;active catalog items vs. items actually sold</span>

Reason: Flags unsold inventory that may need discounts or removal.

In [ ]:
SELECT ProductID, Name
FROM Production.Product
WHERE SellEndDate IS NULL
EXCEPT
SELECT DISTINCT sod.ProductID, p.Name
FROM Sales.SalesOrderDetail sod
JOIN Sales.SalesOrderHeader soh 
    ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product p 
    ON sod.ProductID = p.ProductID
WHERE YEAR(soh.OrderDate) = 2014;


P9: Customers who purchased in both 2013 and 2014.

<span style="color: var(--vscode-foreground);">Spec:&nbsp;</span>  <span style="color: var(--vscode-foreground);">Detect the latest two sales years from SalesOrderHeader.OrderDate.&nbsp;</span>   <span style="color: var(--vscode-foreground);">Get customer IDs for each year.&nbsp;</span> <span style="color: var(--vscode-foreground);">Use INTERSECT to show only those present in both years.</span>

Reason: Identifies repeat customers for loyalty and retention programs.

In [ ]:
SELECT DISTINCT CustomerID
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) = 2013
INTERSECT
SELECT DISTINCT CustomerID
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) = 2014;

P10: Ship to cities active in December and January

Spec: <span style="color: var(--vscode-foreground);">&nbsp;INTERSECT isolates recurring shippin destination</span>

Reason: Shows stable regions with steady demand for better shipping and stock planning.<span style="color: var(--vscode-foreground);"><br></span>

In [ ]:
SELECT DISTINCT a.City
FROM Sales.SalesOrderHeader soh
JOIN Person.Address a 
    ON soh.ShipToAddressID = a.AddressID
WHERE MONTH(soh.OrderDate) = 12
INTERSECT
SELECT DISTINCT a.City
FROM Sales.SalesOrderHeader soh
JOIN Person.Address a 
    ON soh.ShipToAddressID = a.AddressID
WHERE MONTH(soh.OrderDate) = 1;